In [0]:
from pyspark.sql.functions import col, when

train = spark.table("ws1.default.bronze_train")
test  = spark.table("ws1.default.bronze_test")

def transform(df):
    
    return (
        df.withColumn(
            "Gender",
            when(col("Gender") == "Male", 1).otherwise(0)
        )
        .withColumn(
            "Vehicle_Damage",
            when(col("Vehicle_Damage") == "Yes", 1).otherwise(0)
        )
        .withColumn(
            "Vehicle_Age",
            when(col("Vehicle_Age") == "< 1 Year", 0)
            .when(col("Vehicle_Age") == "1-2 Year", 1)
            .otherwise(2)
        )
    )

train_clean = transform(train)
test_clean = transform(test)

train_clean.write.format("delta").mode("overwrite").saveAsTable("ws1.default.gold_train")
test_clean.write.format("delta").mode("overwrite").saveAsTable("ws1.default.gold_test")

print("Gold tables ready")